# 12 — Intelligent Decision Layer with `ai_decide`

In the previous notebooks, our Investment AI system learned how to retrieve research, answer structured portfolio questions, route work across specialized capabilities, and enforce deterministic portfolio-risk policy.

Now we add a different kind of intelligence:

> **Judgment across multiple signals.**

Suppose our system already knows:

```text
Company: Synthetic Compute Company 01
Research signal: Full-year outlook was reduced
Portfolio weight: 15%
Risk policy: BREACH
Excess over concentration limit: 5%
```

The concentration rule itself is deterministic and belongs in our Unity Catalog function.

But a different question is:

> **Given all of these signals together, should this situation be escalated for human review, what action is appropriate, and how urgent is it?**

That is the problem we demonstrate with `ai_decide`.

```text
Research Signal + Portfolio Exposure + Governed Risk Result
                         ↓
                Combined Decision State
                         ↓
                     ai_decide
                         ↓
      Probability + Recommended Action + Urgency
                         ↓
                Human / Analyst Workflow
```


It adds a structured judgment layer after the relevant facts have already been collected.

## 1. Important `ai_decide` context

`ai_decide` evaluates one shared `state` against one or more clearly defined questions.

For this notebook we use all three supported decision styles:

- **`noul`** — returns a probability for a true/false judgment,
- **`choice`** — chooses from named actions,
- **`score`** — evaluates an ordered severity or urgency scale.

`ai_decide` is currently a **Beta** feature, so supported compute and regional availability are required.

## 2. Build the combined decision state

For this notebook, we use a representative state that combines the kinds of outputs our existing capabilities can produce.

This keeps the notebook focused on `ai_decide` itself.

Later, the Multi-Agent Orchestrator can construct the same state dynamically from the Research Agent, Portfolio Agent, and governed risk tool.

In [0]:
import json

decision_state = {
    "security_id": "IA-SYN-01",
    "company_name": "Synthetic Compute Company 01",
    "research": {
        "guidance_reduced": True,
        "period": "Q2 2026",
        "summary": (
            "The company lowered its full-year outlook. "
            "The retrieved research cited export-license timing, "
            "a delayed hyperscale qualification cycle, and a less favorable "
            "second-half product mix."
        ),
    },
    "portfolio": {
        "portfolio_value_usd": 100_000_000,
        "position_value_usd": 15_000_000,
        "position_weight": 0.15,
        "as_of_date": "2026-06-30",
    },
    "risk_policy": {
        "concentration_threshold": 0.10,
        "risk_status": "BREACH",
        "excess_weight": 0.05,
    },
}

print(json.dumps(decision_state, indent=2))

## 3. Define the decisions we want from `ai_decide`

All questions below evaluate the **same state**.

We ask:

1. Does this situation need escalation? → probability
2. What should happen next? → choice
3. How urgent is it? → ordered score

The important point is that we define the decision rubric explicitly instead of asking for an open-ended LLM answer.

In [0]:
decision_questions = {
    "needs_escalation": {
        "type": "noul",
        "instructions": (
            "Based only on the supplied investment state, does this situation "
            "require human review because the combination of research deterioration "
            "and portfolio risk is materially significant?"
        ),
        "criteria": {
            "true": (
                "The state shows meaningful deterioration or uncertainty together "
                "with material portfolio exposure, a policy breach, or another "
                "signal that warrants human attention."
            ),
            "false": (
                "The signals are routine, immaterial, within policy, or do not "
                "justify human review."
            ),
        },
    },
    "recommended_action": {
        "type": "choice",
        "instructions": "Choose the most appropriate next action for this investment situation.",
        "criteria": {
            "monitor": "Continue normal monitoring.",
            "analyst_review": "Send the situation to an investment analyst for focused review.",
            "immediate_review": "Escalate for prompt human review because multiple material signals indicate elevated risk.",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgently should a human review this investment situation?",
        "criteria": [
            "Routine: normal monitoring is sufficient.",
            "Attention needed: an analyst should review the situation soon.",
            "High priority: the situation should receive prompt human review.",
        ],
    },
}

print(json.dumps(decision_questions, indent=2))

In [0]:
# Pre-flight check: verify ai_decide is available in this workspace

try:
    test_result = spark.sql("""
        SELECT ai_decide(
          '{"event": "Portfolio position exceeds internal risk limit"}',
          '{
            "needs_review": {
              "type": "noul",
              "instructions": "Does this event require human review?",
              "criteria": {
                "true": "The event represents a meaningful risk condition requiring attention.",
                "false": "The event is routine and does not require human attention."
              }
            }
          }',
          map('version', '1.0')
        ) AS decision
    """)

    display(test_result)

    print("✅ ai_decide is available in this workspace.")

except Exception as e:
    print("❌ ai_decide is not currently available in this workspace.")
    print()
    print("Possible reasons:")
    print("- ai_decide is not enabled in this region")
    print("- Free Edition AI Runtime access is limited")
    print("- Workspace preview access is not enabled")
    print("- The current compute/runtime does not support ai_decide")
    print()
    print("Databricks error:")
    print(e)

## 4. Call `ai_decide`

The function takes:

```text
ai_decide(state, questions, options)
```

The state contains the facts and context.

The questions define the judgments we want.

One `ai_decide` call can evaluate all three questions against the same investment state.

In [0]:
state_json = json.dumps(decision_state).replace("'", "''")
questions_json = json.dumps(decision_questions).replace("'", "''")

decision_df = spark.sql(
    f'''
    SELECT ai_decide(
        '{state_json}',
        '{questions_json}',
        map('version', '1.0')
    ) AS decision
    '''
)

display(decision_df)
decision_df.createOrReplaceTempView("investment_ai_decision_result")

## 5. Extract the structured decision outputs

`ai_decide` returns a `VARIANT`.

Instead of treating the result as free-form text, we extract the fields our application needs:

- escalation probability,
- recommended action,
- action confidence,
- urgency score,
- urgency confidence.

In [0]:
decision_summary = spark.sql(
    '''
    SELECT
        try_variant_get(
            decision,
            '$.response.answers.needs_escalation.probability',
            'double'
        ) AS escalation_probability,

        try_variant_get(
            decision,
            '$.response.answers.recommended_action.choice',
            'string'
        ) AS recommended_action,

        try_variant_get(
            decision,
            '$.response.answers.recommended_action.confidence',
            'double'
        ) AS action_confidence,

        try_variant_get(
            decision,
            '$.response.answers.urgency.score',
            'double'
        ) AS urgency_score,

        try_variant_get(
            decision,
            '$.response.answers.urgency.confidence',
            'double'
        ) AS urgency_confidence,

        try_variant_get(
            decision,
            '$.error_message',
            'string'
        ) AS error_message

    FROM investment_ai_decision_result
    '''
)

display(decision_summary)
decision_summary.createOrReplaceTempView("investment_ai_decision_summary")

## 6. Turn the AI judgment into an operational next step

A decision becomes useful when an application can act on it.

Here we demonstrate a simple downstream routing rule:

- `immediate_review` or very high escalation probability → **Human Review Queue**
- `analyst_review` → **Analyst Review Queue**
- otherwise → **Continue Monitoring**

`ai_decide` makes the structured judgment.

Our application still controls what action is taken from that judgment.

In [0]:
display(
    spark.sql(
        '''
        SELECT
            escalation_probability,
            recommended_action,
            action_confidence,
            urgency_score,
            urgency_confidence,
            CASE
                WHEN recommended_action = 'immediate_review'
                     OR escalation_probability >= 0.80
                    THEN 'HUMAN_REVIEW_QUEUE'
                WHEN recommended_action = 'analyst_review'
                    THEN 'ANALYST_REVIEW_QUEUE'
                ELSE 'CONTINUE_MONITORING'
            END AS operational_next_step
        FROM investment_ai_decision_summary
        '''
    )
)

# ✅ Notebook 12 complete

We now have four clearly separated capability patterns:

```text
Research evidence
Research Agent → AI Search

Flexible portfolio analytics
Portfolio Agent → Genie + Metric View

Exact business policy
Risk Tool → Unity Catalog Function

Judgment across multiple signals
Decision Layer → ai_decide
```

The key distinction is:

> **A UC function answers an exact rule. `ai_decide` makes a structured judgment from multiple signals.**

For example:

```text
15% > 10%
```

is a deterministic concentration-policy calculation.

But:

```text
The company lowered guidance
+ we own 15%
+ the position breached concentration policy
→ should a human review this urgently?
```

is a judgment problem.

That is where `ai_decide` fits.

We do **not** need to redesign Notebook 10.

Later, the orchestrator can gather the relevant research, portfolio, and risk outputs and pass the combined state into this decision layer when a judgment is needed.

### Next

**Notebook 13 — MLflow Agent Evaluation and Quality**